In [1]:
# !pip install gensim
# !pip install python-Levenshtein

In [2]:
import gensim
import pandas as pd

### Reading and Exploring the Dataset
The dataset we are using here is a subset of Amazon reviews from the Cell Phones & Accessories category. The data is stored as a JSON file and can be read using pandas.

Link to the Dataset: http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Cell_Phones_and_Accessories_5.json.gz

In [3]:
df = pd.read_json("reviews_Cell_Phones_and_Accessories_5.json", lines=True)
df

FileNotFoundError: File reviews_Cell_Phones_and_Accessories_5.json does not exist

In [4]:
df.shape

(194439, 9)

### Simple Preprocessing & Tokenization
The first thing to do for any data science task is to clean the data.
For NLP, we apply various processing like converting all the words to lower case, trimming spaces, removing punctuations. 
This is something we will do over here too.

Additionally, we can also remove stop words like 'and', 'or', 'is', 'the', 'a', 'an' and convert words to their root forms like 'running' to 'run'.

In [5]:
review_text = df.reviewText.apply(gensim.utils.simple_preprocess)

In [6]:
review_text

0         [they, look, good, and, stick, good, just, don...
1         [these, stickers, work, like, the, review, say...
2         [these, are, awesome, and, make, my, phone, lo...
3         [item, arrived, in, great, time, and, was, in,...
4         [awesome, stays, on, and, looks, great, can, b...
                                ...                        
194434    [works, great, just, like, my, original, one, ...
194435    [great, product, great, packaging, high, quali...
194436    [this, is, great, cable, just, as, good, as, t...
194437    [really, like, it, becasue, it, works, well, w...
194438    [product, as, described, have, wasted, lot, of...
Name: reviewText, Length: 194439, dtype: object

In [7]:
review_text.loc[0]

['they',
 'look',
 'good',
 'and',
 'stick',
 'good',
 'just',
 'don',
 'like',
 'the',
 'rounded',
 'shape',
 'because',
 'was',
 'always',
 'bumping',
 'it',
 'and',
 'siri',
 'kept',
 'popping',
 'up',
 'and',
 'it',
 'was',
 'irritating',
 'just',
 'won',
 'buy',
 'product',
 'like',
 'this',
 'again']

In [8]:
df.reviewText.loc[0]

"They look good and stick good! I just don't like the rounded shape because I was always bumping it and Siri kept popping up and it was irritating. I just won't buy a product like this again"

### Training the Word2Vec Model

Train the model for reviews. Use a window of size 10 i.e. 10 words before the present word and 10 words ahead. A sentence with at least 2 words should only be considered, configure this using min_count parameter.

Workers define how many CPU threads to be used.

#### Initialize the model

In [ ]:

model = gensim.models.Word2Vec(
    window=10,
    min_count=2,
    workers=2,
)



#### Build Vocabulary

In [10]:
model.build_vocab(review_text, progress_per=1000)

#### Train the Word2Vec Model

In [11]:
model.train(review_text, total_examples=model.corpus_count, epochs=model.epochs)

(61506100, 83868975)

### Save the Model

Save the model so that it can be reused in other applications

In [12]:
model.save("./word2vec-amazon-cell-accessories-reviews-short.model")

### Finding Similar Words and Similarity between words
https://radimrehurek.com/gensim/models/word2vec.html

In [13]:
model.wv.most_similar("bad")

[('terrible', 0.6910141110420227),
 ('shabby', 0.6390656232833862),
 ('horrible', 0.6069936156272888),
 ('good', 0.5785307884216309),
 ('funny', 0.5541638135910034),
 ('awful', 0.5375000834465027),
 ('crappy', 0.5371657609939575),
 ('disappointing', 0.5261415839195251),
 ('poor', 0.5128635764122009),
 ('keen', 0.5045686960220337)]

In [14]:
model.wv.similarity(w1="cheap", w2="inexpensive")

0.51952136

In [15]:
model.wv.similarity(w1="great", w2="good")

0.7792832

In [16]:
import torch
import torch.nn as nn

# Define encoder RNN with a specific hidden size
hidden_size = 256
encoder_rnn = nn.LSTM(input_size=128, hidden_size=hidden_size, num_layers=1, bidirectional=False)

# Output hidden state (context vector) will be of size [hidden_size]
input_seq = torch.randn(10, 1, 128)  # Example input sequence of 10 steps
_, (hidden_state, _) = encoder_rnn(input_seq)

context_vector = hidden_state[-1]  # The last hidden state
print(context_vector.shape)  # Should print: torch.Size([1, 256])

torch.Size([1, 256])


In [17]:
class SelfAttention(torch.nn.Module):
	def __init__(self, d_model):
		super(SelfAttention, self).__init__()
		self.d_model = d_model
		self.number_of_heads = 1
		self.W_q = torch.nn.Linear(d_model, d_model)
		self.W_k = torch.nn.Linear(d_model, d_model)
		self.W_v = torch.nn.Linear(d_model, d_model)
		self.W_o = torch.nn.Linear(d_model, d_model)

	def forward(self, X):
		Q = self.W_q(X)
		K = self.W_k(X)
		V = self.W_v(X)
		d_k = Q.size(-1) // self.number_of_heads
		attention_scores = torch.matmul(Q, K.transpose(-2, -1)) / (d_k ** 0.5)
		attention_weights = torch.softmax(attention_scores, dim=-1)
		output = torch.matmul(attention_weights, V)
		output = self.W_o(output)
		return output

# Example usage
d_model = 4
self_attention = SelfAttention(d_model)
X = torch.tensor([[1.0, 0.5, 0.2, 0.7], [0.2, 0.4, 0.6, 0.8]])
output = self_attention(X)
print(output)

tensor([[ 0.6616, -0.1363,  0.3944, -0.1927],
        [ 0.6704, -0.1366,  0.3966, -0.2004]], grad_fn=<AddmmBackward0>)


In [21]:
def add(a: int, b: int) -> int:
    return a + b

# Usage
result = add(3, 4)  # Returns 7
print(result)  # Output: 7
print(add.__annotations__)  # Output: {'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}

7
{'a': <class 'int'>, 'b': <class 'int'>, 'return': <class 'int'>}


### Further Reading

You can read about gensim more at https://radimrehurek.com/gensim/models/word2vec.html

Explore other Datasets related to Amazon Reviews: http://jmcauley.ucsd.edu/data/amazon/

## Exercise

Train a word2vec model on the [Sports & Outdoors Reviews Dataset](http://snap.stanford.edu/data/amazon/productGraph/categoryFiles/reviews_Sports_and_Outdoors_5.json.gz)
Once you train a model on this, find the words most similar to 'awful' and find similarities between the following word tuples: ('good', 'great'), ('slow','steady')

Click here for [solution](https://github.com/codebasics/deep-learning-keras-tf-tutorial/blob/master/42_word2vec_gensim/42_word2vec_gensim_exercise_solution.ipynb).